In [1]:
"""
CarbonTatva AI — Model: Linear Regression + Ridge
==================================================
Predicts next-day (96 slots = 24h) electricity prices
separately for GDAM, DAM, RTM markets.

Run:
    python model_ridge.py

Output: outputs_ridge/
"""

import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

warnings.filterwarnings('ignore')
os.makedirs('outputs_ridge', exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
DATA_FILE  = 'final_merged_data2.xlsx'
MARKETS    = ['GDAM', 'DAM', 'RTM']
SLOTS_DAY  = 96
TRAIN_END  = '2024-12-31'
VAL_END    = '2025-09-30'
SEED       = 42
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────
def build_features(df):
    df = df.copy()

    # Parse datetime
    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    # Drop collinear columns
    df.drop(columns=['apparent_temperature (°C)',
                     'sunshine_duration (s)',
                     'precipitation (mm)'], errors='ignore', inplace=True)

    # ── Temporal ──────────────────────────────────────────────
    df['hour']  = df['datetime'].dt.hour
    df['dow']   = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month

    # Cyclical sin/cos — wraps 23h→0h correctly
    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)

    # Holiday flag
    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        df['holiday_flag'] = 0

    # ── Autoregressive ────────────────────────────────────────
    p = df['Price (Rs./MWh)']
    df['price_lag96']  = p.shift(96)     # same slot yesterday
    df['price_lag672'] = p.shift(672)    # same slot last week
    df['ma3d']  = p.shift(1).rolling(288).mean()
    df['ma7d']  = p.shift(1).rolling(672).mean()
    df['ma14d'] = p.shift(1).rolling(1344).mean()

    # Price momentum features (helps Ridge capture trends)
    df['price_diff1d'] = p.shift(96) - p.shift(192)   # yesterday vs day before
    df['price_diff1w'] = p.shift(672) - p.shift(1344) # last week vs week before

    # ── Weather ───────────────────────────────────────────────
    df['temp']    = df['temperature_2m (°C)']
    df['temp_sq'] = df['temp'] ** 2
    df['temp_cb'] = df['temp'] ** 3                    # cubic for sharper extremes

    df['radiation']             = df['shortwave_radiation (W/m²)']
    df['radiation_x_hour_sin']  = df['radiation'] * df['hour_sin']
    df['radiation_x_hour_cos']  = df['radiation'] * df['hour_cos']
    df['radiation_sq']          = df['radiation'] ** 2  # non-linear solar effect

    df['wind_speed']  = df['wind_speed_100m (km/h)']
    df['wind_sq']     = df['wind_speed'] ** 2
    df['cloud_cover'] = df['cloud_cover (%)']
    df['humidity']    = df['relative_humidity_2m (%)']
    df['rain']        = df['rain (mm)']

    # Interaction features (key for Ridge — it can't learn these automatically)
    df['temp_x_hour_sin']   = df['temp'] * df['hour_sin']
    df['temp_x_is_weekend'] = df['temp'] * df['is_weekend']
    df['wind_x_radiation']  = df['wind_speed'] * df['radiation']

    return df


FEATURES = [
    # Temporal
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
    'is_weekend', 'holiday_flag',
    # Autoregressive
    'price_lag96', 'price_lag672',
    'ma3d', 'ma7d', 'ma14d',
    'price_diff1d', 'price_diff1w',
    # Weather — base
    'temp', 'temp_sq', 'temp_cb',
    'radiation', 'radiation_sq',
    'radiation_x_hour_sin', 'radiation_x_hour_cos',
    'wind_speed', 'wind_sq',
    'cloud_cover', 'humidity', 'rain',
    # Interactions
    'temp_x_hour_sin', 'temp_x_is_weekend', 'wind_x_radiation',
]

# ─────────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"    {label:20s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:.2f}%")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def save_pred_plot(y_true, y_pred, title, path):
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27')
        ax.tick_params(colors='#aaa')

    # Full test
    n = min(800, len(y_true))
    axes[0].plot(y_true[:n], color='#aaa', lw=0.8, label='Actual')
    axes[0].plot(y_pred[:n], color='#8e44ad', lw=0.8, label='Predicted')
    axes[0].set_title(f'{title} — Test Set', color='white', fontsize=12)
    axes[0].legend(fontsize=9)
    axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')

    # 3-day zoom
    n2 = SLOTS_DAY * 3
    axes[1].plot(y_true[:n2], color='#aaa', lw=1.5, label='Actual')
    axes[1].plot(y_pred[:n2], color='#8e44ad', lw=1.5, ls='--', label='Predicted')
    axes[1].set_title(f'{title} — First 3 Days (zoomed)', color='white', fontsize=12)
    axes[1].legend(fontsize=9)
    axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')

    # Scatter: actual vs predicted
    axes[2].scatter(y_true[:3000], y_pred[:3000],
                    s=2, alpha=0.3, color='#8e44ad')
    lim = max(y_true[:3000].max(), y_pred[:3000].max())
    axes[2].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect prediction')
    axes[2].set_title(f'{title} — Actual vs Predicted Scatter', color='white', fontsize=12)
    axes[2].set_xlabel('Actual Price', color='#aaa')
    axes[2].set_ylabel('Predicted Price', color='#aaa')
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


def save_coef_plot(coefs, feature_names, title, path):
    idx = np.argsort(np.abs(coefs))
    bar_colors = ['#e74c3c' if c > 0 else '#2980b9' for c in coefs[idx]]

    fig, ax = plt.subplots(figsize=(11, 9))
    fig.patch.set_facecolor('#0f1117')
    ax.set_facecolor('#1a1d27')
    ax.barh([feature_names[i] for i in idx], coefs[idx],
            color=bar_colors, alpha=0.85)
    ax.axvline(0, color='white', lw=0.8, ls='--')
    ax.set_title(f'{title}\nRed = increases price  |  Blue = decreases price',
                 color='white', fontsize=11)
    ax.tick_params(colors='#aaa')
    ax.set_xlabel('Coefficient value', color='#aaa')
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


# ─────────────────────────────────────────────────────────────────
# MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("MODEL: Ridge Regression")
print("=" * 60)
print(f"Loading data from {DATA_FILE}...")
sheets = pd.read_excel(DATA_FILE, sheet_name=None)

all_metrics = {}

for mkt in MARKETS:
    print(f"\n{'─'*50}")
    print(f"Market: {mkt}")
    print('─' * 50)

    df = build_features(sheets[mkt])
    df = df.dropna(subset=FEATURES + ['Price (Rs./MWh)']).copy()

    # Split
    train = df[df['datetime'] <= TRAIN_END]
    val   = df[(df['datetime'] > TRAIN_END) & (df['datetime'] <= VAL_END)]
    test  = df[df['datetime'] > VAL_END]
    print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

    X_tr = train[FEATURES].values;  y_tr = train['Price (Rs./MWh)'].values
    X_va = val[FEATURES].values;    y_va = val['Price (Rs./MWh)'].values
    X_te = test[FEATURES].values;   y_te = test['Price (Rs./MWh)'].values

    # Scale — fit ONLY on train
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    X_te_s = scaler.transform(X_te)

    # ── Grid search alpha on validation ──────────────────────
    print(f"  Tuning alpha...")
    best_alpha, best_mae = None, float('inf')
    alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 50.0,
              100.0, 500.0, 1000.0, 5000.0, 10000.0]

    for alpha in alphas:
        m = Ridge(alpha=alpha, max_iter=10000)
        m.fit(X_tr_s, y_tr)
        val_mae = mean_absolute_error(y_va, m.predict(X_va_s))
        if val_mae < best_mae:
            best_mae   = val_mae
            best_alpha = alpha

    print(f"  Best alpha: {best_alpha}  |  Val MAE: {best_mae:.1f}")

    # ── Final model on train+val combined ─────────────────────
    # Re-fit on train+val for final test evaluation
    X_trainval = np.vstack([X_tr_s, X_va_s])
    y_trainval = np.concatenate([y_tr, y_va])

    # But scale on train only — already done
    # Just append val scaled data
    final_model = Ridge(alpha=best_alpha, max_iter=10000)
    final_model.fit(X_trainval, y_trainval)

    # ── Evaluate ──────────────────────────────────────────────
    val_pred  = np.clip(final_model.predict(X_va_s), 0, 20000)
    test_pred = np.clip(final_model.predict(X_te_s), 0, 20000)

    print(f"  Results:")
    m_val  = compute_metrics(y_va, val_pred,  'Validation')
    m_test = compute_metrics(y_te, test_pred, 'Test')
    all_metrics[mkt] = {'val': m_val, 'test': m_test}

    # ── Plots ─────────────────────────────────────────────────
    save_pred_plot(y_te, test_pred,
                   f'Ridge — {mkt}',
                   f'outputs_ridge/ridge_pred_{mkt}.png')

    save_coef_plot(final_model.coef_, FEATURES,
                   f'Ridge Coefficients — {mkt}',
                   f'outputs_ridge/ridge_coefs_{mkt}.png')

    # ── Feature correlation with price (extra insight) ────────
    corr_vals = [np.corrcoef(train[f].values, y_tr)[0, 1]
                 for f in FEATURES]
    corr_idx = np.argsort(np.abs(corr_vals))

    fig, ax = plt.subplots(figsize=(11, 9))
    fig.patch.set_facecolor('#0f1117')
    ax.set_facecolor('#1a1d27')
    bar_c = ['#e74c3c' if c > 0 else '#2980b9' for c in
             [corr_vals[i] for i in corr_idx]]
    ax.barh([FEATURES[i] for i in corr_idx],
            [corr_vals[i] for i in corr_idx],
            color=bar_c, alpha=0.85)
    ax.axvline(0, color='white', lw=0.8, ls='--')
    ax.set_title(f'Feature Correlation with Price — {mkt}',
                 color='white', fontsize=11)
    ax.tick_params(colors='#aaa')
    ax.set_xlabel('Pearson r', color='#aaa')
    plt.tight_layout()
    plt.savefig(f'outputs_ridge/ridge_corr_{mkt}.png',
                dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    # ── Save model ────────────────────────────────────────────
    with open(f'outputs_ridge/ridge_{mkt}.pkl', 'wb') as f:
        pickle.dump({'model': final_model,
                     'scaler': scaler,
                     'features': FEATURES,
                     'best_alpha': best_alpha}, f)
    print(f"  Saved: outputs_ridge/ridge_{mkt}.pkl")


# ─────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY — Ridge Regression")
print('=' * 60)

rows = []
for mkt, res in all_metrics.items():
    for split, m in res.items():
        rows.append({'Market': mkt, 'Split': split,
                     'MAE': round(m['mae'], 1),
                     'RMSE': round(m['rmse'], 1),
                     'MAPE%': round(m['mape'], 2)})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('outputs_ridge/metrics.csv', index=False)

# Comparison chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Ridge Regression — Test MAE by Market', color='white', fontsize=14)
for ax, mkt in zip(axes, MARKETS):
    ax.set_facecolor('#1a1d27')
    if mkt in all_metrics:
        mae = all_metrics[mkt]['test']['mae']
        ax.bar([mkt], [mae], color='#8e44ad', alpha=0.85)
        ax.text(0, mae + 5, f'₹{mae:.0f}', ha='center',
                color='white', fontsize=12, fontweight='bold')
    ax.set_title(mkt, color='white')
    ax.set_ylabel('MAE (Rs./MWh)', color='#aaa')
    ax.tick_params(colors='#aaa')
plt.tight_layout()
plt.savefig('outputs_ridge/summary_chart.png',
            dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.close()

print("\n✅ Done! Check outputs_ridge/")

MODEL: Ridge Regression
Loading data from final_merged_data.xlsx...

──────────────────────────────────────────────────
Market: GDAM
──────────────────────────────────────────────────
  Train: 102,721 | Val: 26,208 | Test: 14,591
  Tuning alpha...
  Best alpha: 500.0  |  Val MAE: 692.5
  Results:
    Validation            MAE=  670.7  RMSE= 1001.8  MAPE=17.89%
    Test                  MAE=  476.3  RMSE=  769.1  MAPE=13.89%
  Saved: outputs_ridge/ridge_GDAM.pkl

──────────────────────────────────────────────────
Market: DAM
──────────────────────────────────────────────────
  Train: 102,721 | Val: 26,208 | Test: 14,591
  Tuning alpha...
  Best alpha: 500.0  |  Val MAE: 727.5
  Results:
    Validation            MAE=  708.7  RMSE= 1108.6  MAPE=24.03%
    Test                  MAE=  523.0  RMSE=  839.8  MAPE=28.81%
  Saved: outputs_ridge/ridge_DAM.pkl

──────────────────────────────────────────────────
Market: RTM
──────────────────────────────────────────────────
  Train: 101,973 | Val: